# 03 — Model Training
Train ALS (Collaborative Filtering), Content-Based, GRU4Rec (Session-Based),
ConversionRanker, and assemble the Hybrid Recommender.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), "src"))

import pandas as pd
import scipy.sparse as sp
import pickle

PROJECT_ROOT = os.path.dirname(os.getcwd())
PROCESSED    = f"{PROJECT_ROOT}/data/processed"
MODELS_DIR   = f"{PROJECT_ROOT}/outputs/models"
os.makedirs(MODELS_DIR, exist_ok=True)
print("Dirs ready.")

## 1. Load Processed Features

In [ ]:
interactions = pd.read_parquet(f"{PROCESSED}/interactions.parquet")
sequences    = pd.read_parquet(f"{PROCESSED}/session_sequences.parquet")
tfidf_mat    = sp.load_npz(f"{PROCESSED}/tfidf_matrix.npz")
item_ids     = pd.read_csv(f"{PROCESSED}/tfidf_item_ids.csv").squeeze()

with open(f"{PROCESSED}/tfidf_vectorizer.pkl", "rb") as f:
    vectorizer = pickle.load(f)

print(f"Interactions : {len(interactions):,}")
print(f"Sequences    : {len(sequences):,}")
print(f"TF-IDF matrix: {tfidf_mat.shape}")

## 2. Train ALS Collaborative Filtering

In [ ]:
from models.collaborative_filtering import ALSRecommender

als = ALSRecommender(factors=128, regularization=0.01, iterations=20, alpha=40)
als.fit(interactions)
als.save(f"{MODELS_DIR}/als_model.pkl")
print("ALS model saved.")

## 3. Train Content-Based Recommender

In [ ]:
from models.content_based import ContentBasedRecommender

cb = ContentBasedRecommender(top_k_similar=50)
cb.fit(item_ids, tfidf_mat, vectorizer)
cb.save(f"{MODELS_DIR}/content_based_model.pkl")
print("Content-Based model saved.")

## 4. Train Session-Based Recommender (GRU4Rec)

In [ ]:
from models.session_based import SessionBasedRecommender

sb = SessionBasedRecommender(emb_dim=64, hidden_size=128, dropout=0.2, max_session_length=20)
sb.fit(sequences, epochs=20, batch_size=256, lr=0.001)
sb.save(f"{MODELS_DIR}/session_based_model.pkl")
print("GRU4Rec model saved.")

## 5. Assemble Hybrid Recommender

In [ ]:
from models.hybrid import HybridRecommender

hybrid = HybridRecommender(
    cf_model=als,
    cb_model=cb,
    sb_model=sb,
    weights={"collaborative_filtering": 0.35, "content_based": 0.25, "session_based": 0.40},
)

# Smoke-test: recommend for a sample session
sample_session = sequences["item_sequence"].iloc[0]
sample_session = [i for i in sample_session if i != 0]
recs = hybrid.recommend(
    visitor_id=sequences["visitorid"].iloc[0],
    session_items=sample_session,
    interactions=interactions,
    top_k=10,
)
print("Sample hybrid recommendations:")
for rank, (item, score) in enumerate(recs, 1):
    print(f"  {rank:2d}. item {item:>8d}  score={score:.4f}")